# AutoEncoder Reconstruction

这个 notebook 用来学习 AutoEncoder 图片重建任务。

AutoEncoder 的目标是：

```text
input image -> reconstructed image
```

这一步是从分类任务过渡到生成任务的基础练习。

## 这一步要学会什么

- 知道模型不只可以输出类别，也可以输出图片
- 理解 encoder、decoder 和 latent feature 的基本作用
- 跑通一个最小 image-to-image 训练流程
- 显示原图和重建图的对比


## 1. Import Libraries

先导入需要的库，并检查 PyTorch 是否可以正常使用。


In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


## 2. Load Fashion MNIST

这里加载 Fashion MNIST 数据集。

当前阶段重点不是追求模型效果，而是熟悉数据加载流程。


In [ ]:
transform = transforms.ToTensor()

train_dataset = torchvision.datasets.FashionMNIST(
    root="../data/raw",
    train=True,
    download=True,
    transform=transform
)

print("Number of training samples:", len(train_dataset))


## 3. Inspect One Image

先查看单张图片的 shape。

Fashion MNIST 是灰度图，所以单张图片 shape 通常是：

```text
[1, 28, 28]
```

含义是：

```text
1: 灰度通道
28: 图片高度
28: 图片宽度
```


In [ ]:
image, label = train_dataset[0]

print("Image shape:", image.shape)
print("Label:", label)


In [ ]:
plt.imshow(image.squeeze(), cmap="gray")
plt.title(f"Label: {label}")
plt.axis("off")
plt.show()


## 4. DataLoader

DataLoader 用来把数据按 batch 送进模型。

单张图片是：

```text
[1, 28, 28]
```

一个 batch 是：

```text
[batch_size, 1, 28, 28]
```


In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

images, labels = next(iter(train_loader))

print("Batch image shape:", images.shape)
print("Batch label shape:", labels.shape)


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(8, 4))

for i, ax in enumerate(axes.flatten()):
    ax.imshow(images[i].squeeze(), cmap="gray")
    ax.set_title(f"Label: {labels[i].item()}")
    ax.axis("off")

plt.tight_layout()
plt.show()


## 5. Build AutoEncoder

AutoEncoder 的结构可以理解为：

```text
input image -> encoder -> latent feature -> decoder -> reconstructed image
```

在这个任务中，模型的目标是让 reconstructed image 尽量接近 input image。


In [ ]:
import torch.nn as nn

class SimpleAutoEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 32),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.Linear(32, 128),
            nn.ReLU(),
            nn.Linear(128, 28 * 28),
            nn.Sigmoid()
        )

    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        reconstructed = reconstructed.view(-1, 1, 28, 28)
        return reconstructed


## 6. Create Model, Loss, and Optimizer

这里使用：

- MSELoss：比较原图和重建图的像素差距
- Adam：根据 loss 更新模型参数


In [ ]:
model = SimpleAutoEncoder()

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(model)


## 7. Train AutoEncoder

训练时，输入图片和目标图片是同一张图。

```text
input = image
target = image
```

CPU 环境下先训练 1 个 epoch，跑通流程最重要。


In [ ]:
epochs = 1

for epoch in range(epochs):
    total_loss = 0.0

    for images, _ in train_loader:
        outputs = model(images)
        loss = criterion(outputs, images)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch + 1}/{epochs}], Loss: {avg_loss:.4f}")


## 8. Visualize Reconstruction Results

第一行是原图，第二行是 AutoEncoder 重建出来的图。

如果训练 1 个 epoch 后图片比较模糊，这是正常现象。当前目标是跑通流程，不是追求高质量结果。


In [ ]:
model.eval()

images, _ = next(iter(train_loader))

with torch.no_grad():
    reconstructed = model(images)

fig, axes = plt.subplots(2, 8, figsize=(12, 4))

for i in range(8):
    axes[0, i].imshow(images[i].squeeze(), cmap="gray")
    axes[0, i].axis("off")
    axes[0, i].set_title("Original")

    axes[1, i].imshow(reconstructed[i].squeeze(), cmap="gray")
    axes[1, i].axis("off")
    axes[1, i].set_title("Recon")

plt.tight_layout()
plt.show()


## 9. Save Result Figure

把原图和重建图的对比保存到：

```text
outputs/figures/autoencoder_reconstruction.png
```


In [ ]:
from pathlib import Path

output_dir = Path("../outputs/figures")
output_dir.mkdir(parents=True, exist_ok=True)

save_path = output_dir / "autoencoder_reconstruction.png"
fig.savefig(save_path)

print("Saved to:", save_path)


## 本 notebook 学到什么

完成这个 notebook 后，应能说清楚：

- AutoEncoder 的输入和输出都是图片
- encoder 负责把图片压缩成 latent feature
- decoder 负责把 latent feature 重建成图片
- reconstruction loss 用来衡量重建图和原图的差距
- 这是从 image classification 走向 image generation / prediction 的第一步
